# 🚀 UTPBOT - LANZADOR MAESTRO (VERSIÓN DEFINITIVA)
Este script monta tu Drive, descarga el proyecto, arregla los permisos de seguridad (CORS) y genera un túnel persistente con Cloudflare.

In [ ]:
# =======================================================
# 1. CONFIGURACIÓN INICIAL (Drive y Carpetas)
# =======================================================
from google.colab import drive
import os, subprocess, time, urllib.request, glob, re

drive.mount('/content/drive')
!rm -rf /content/utpbot
!mkdir -p /content/utpbot

# --- BUSCAR Y DESCOMPRIMIR ---
print("📥 Buscando tu archivo utpbot.rar en Drive...")
archivos = glob.glob("/content/drive/MyDrive/**/*utpbot*.[rRzZ][aAiI][rRpP]", recursive=True)

if not archivos:
    raise FileNotFoundError("❌ No encontré el archivo rar/zip en tu Google Drive.")

archivo = archivos[0]
print(f"✅ ¡Archivo ubicado!: {archivo}")

if archivo.lower().endswith(".rar"):
    !unrar x -Y "{archivo}" "/content/utpbot/" > /dev/null 2>&1
else:
    !unzip -q "{archivo}" -d "/content/utpbot/"

# =======================================================
# 2. UBICACIÓN E INSTALACIÓN DE DEPEDENCIAS
# =======================================================
backend_dir = None
for root, dirs, files in os.walk('/content/utpbot'):
    if 'main.py' in files and 'requirements.txt' in files:
        backend_dir = root
        break

if not backend_dir:
    raise FileNotFoundError("❌ El archivo main.py no existe dentro de la carpeta.")

print(f"📂 Backend configurado en: {backend_dir}")
print("⏳ Instalando paquetes... (esto tarda un poco)")
!pip install -q -r "{backend_dir}/requirements.txt"

# =======================================================
# 3. INSTALACIÓN DE CLOUDFLARE (El puente estable)
# =======================================================
print("🌐 Instalando Cloudflare...")
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1

# =======================================================
# 4. PARCHEO DE SEGURIDAD (Arreglo de CORS definitivo)
# =======================================================
main_path = f"{backend_dir}/main.py"
print("🛠 Aplicando parche de seguridad CORS...")

with open(main_path, 'r') as f:
    content = f.read()

# Forzamos que acepte conexiones externas sin errores de seguridad
content = re.sub(r'allow_origins=cors_origins,', 'allow_origins=["*"],', content)
content = re.sub(r'allow_credentials=True,', 'allow_credentials=False,', content)

with open(main_path, 'w') as f:
    f.write(content)

# =======================================================
# 5. LANZAMIENTO DEL SERVIDOR Y TÚNEL PERSISTENTE
# =======================================================
print("\n🚀 Lanzando Backend...")
!pkill uvicorn
!pkill cloudflared

with open("server.log", "w") as log_file:
    subprocess.Popen(
        ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"], 
        cwd=backend_dir,
        stdout=log_file,
        stderr=log_file
    )

time.sleep(10) # Damos tiempo a que el server inicie

print("\n🌐 ACTIVANDO TÚNEL (Mantén esta pestaña de Colab abierta)...\n")
print("---------------------------------------------------------")

conf = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://localhost:8000"], 
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

url_final = None

try:
    while True:
        line = conf.stdout.readline()
        if not line: break
        
        match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
        if match and not url_final:
            url_final = match.group(0)
            print(f"\n✅ ¡AHORA SÍ! LINK LISTO:\n")
            print(f"      {url_final}      \n")
            print(f"👉 Copia este link arriba y actualízalo en tu frontend/script.js")
            print("👉 NO DETENGASesta celda o el bot dejará de responder.")
            print("---------------------------------------------------------")
            
except KeyboardInterrupt:
    print("\n🛑 Apagando el servidor...")
    !pkill uvicorn
!pkill cloudflared